# Mitra Regressor — End-to-End Regression with Your Own Data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/mitra-regressor-pipeline/blob/main/tutorials/mitra_regressor_colab.ipynb)

Use the Mitra Regressor weights distributed through the DIMER Model Repository, or the exact pinned upstream checkpoint as a fallback. If you do not yet have a labelled CSV, use the bundled FreshRetailNet regression sample.

**No DIMER Workbench access is required.** Data is processed in Google Colab, not by DIMER. Do not upload confidential, sensitive, or restricted data unless that environment is permitted.


## 1. Install and inspect the runtime

The Mitra extra pins a compatible PyTorch range and can therefore replace Colab's preinstalled `torch`. The notebook reports the actual installed PyTorch/CUDA versions after installation instead of assuming the accelerator stack was left unchanged. If `torch` was already imported and pip changes its installed version, restart the session before continuing.

Pip may also report conflicts for unrelated preinstalled packages such as `torchvision`, `diffusers`, or `gradio`; this tutorial does not use those packages.


In [ ]:
import importlib.metadata as importlib_metadata
import sys

PREINSTALL_TORCH_VERSION = importlib_metadata.version('torch')
TORCH_WAS_IMPORTED = 'torch' in sys.modules
print('PyTorch before install:', PREINSTALL_TORCH_VERSION)

%pip install -q "autogluon.tabular[mitra]==1.5.0"

INSTALLED_TORCH_VERSION = importlib_metadata.version('torch')
AUTOGLUON_VERSION = importlib_metadata.version('autogluon.tabular')
if TORCH_WAS_IMPORTED and INSTALLED_TORCH_VERSION != PREINSTALL_TORCH_VERSION:
    raise RuntimeError('pip changed PyTorch after it had already been imported. Use Runtime → Restart session, then run the notebook top-to-bottom.')

import torch

TORCH_VERSION = torch.__version__
TORCH_CUDA_VERSION = torch.version.cuda
CUDA_AVAILABLE_AFTER_INSTALL = torch.cuda.is_available()
print('AutoGluon:', AUTOGLUON_VERSION)
print('PyTorch after install:', TORCH_VERSION)
print('PyTorch CUDA build:', TORCH_CUDA_VERSION)
print('CUDA available:', CUDA_AVAILABLE_AFTER_INSTALL)
if not CUDA_AVAILABLE_AFTER_INSTALL:
    print('⚠ Pretrained/in-context evaluation can still run on CPU, but fine-tuning will be disabled.')


## 2. Acquire, verify, and lock the checkpoint

- **DIMER ZIP** — upload the DIMER ZIP containing `model.safetensors`; the notebook retrieves the matching pinned `config.json`.
- **Pinned upstream** — retrieve both files from the exact pinned AutoGluon revision.

Both files are SHA-256 verified and staged into an isolated Hugging Face cache. The notebook then asks Hugging Face to resolve the model exactly as AutoGluon will and refuses to continue unless both resolved files come from that verified snapshot **and still match the expected digests**.

If the lock check fails, use **Runtime → Restart session** and run from Step 1 downward. Network requests use a finite timeout so outages fail clearly instead of hanging indefinitely.


In [ ]:
import hashlib, json, os, random, shutil, urllib.request, zipfile
from pathlib import Path

MODEL_ID = 'autogluon/mitra-regressor'
PINNED_REVISION = '5f277aa8f69042d39d6ac3612aed18bb9279bd95'
EXPECTED_WEIGHTS_SHA256 = 'd8e75c62af0bec2fd404b0ad20a442d951d43ca6d331315cfcc0509b54f2c642'
EXPECTED_CONFIG_SHA256 = '2bc1ed5047f7c25368245e8ad32540a5fa28940b1ec05d3f1f454a09ff5384c1'
NETWORK_TIMEOUT_SECONDS = 30

HF_HOME = Path('/content/mitra-hf')
MODEL_DIR = Path('/content/mitra-model')
HF_HOME.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)
os.environ['HF_HOME'] = str(HF_HOME)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

def fetch_pinned(name, dest):
    url = f'https://huggingface.co/{MODEL_ID}/resolve/{PINNED_REVISION}/{name}?download=true'
    print('Retrieving pinned', name)
    with urllib.request.urlopen(url, timeout=NETWORK_TIMEOUT_SECONDS) as r, open(dest, 'wb') as f:
        shutil.copyfileobj(r, f)

def verify(path, expected, label):
    actual = sha256_file(path)
    if actual != expected:
        raise RuntimeError(f'{label} checksum mismatch.\nExpected: {expected}\nActual:   {actual}')
    print(f'✓ {label} verified: {actual[:12]}…')
    return actual

def weights_from_dimer(dest):
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError('Upload exactly one DIMER ZIP or model.safetensors.')
    name, payload = next(iter(uploaded.items()))
    p = MODEL_DIR / Path(name).name
    p.write_bytes(payload)
    if p.suffix.lower() == '.safetensors':
        if p.resolve() != dest.resolve():
            shutil.copy2(p, dest)
        return
    if p.suffix.lower() != '.zip':
        raise ValueError('Expected a DIMER ZIP or model.safetensors.')
    with zipfile.ZipFile(p) as z:
        matches = [i for i in z.infolist() if not i.is_dir() and Path(i.filename).name == 'model.safetensors']
        if len(matches) != 1:
            raise RuntimeError(f'Expected one model.safetensors in the DIMER ZIP; found {len(matches)}.')
        with z.open(matches[0]) as src, open(dest, 'wb') as dst:
            shutil.copyfileobj(src, dst)

def install_offline_snapshot(weights, config, weights_digest):
    snapshot = weights_digest[:40]
    repo = HF_HOME / 'hub' / ('models--' + MODEL_ID.replace('/', '--'))
    snap = repo / 'snapshots' / snapshot
    refs = repo / 'refs'
    snap.mkdir(parents=True, exist_ok=True)
    refs.mkdir(parents=True, exist_ok=True)
    shutil.copy2(weights, snap / 'model.safetensors')
    shutil.copy2(config, snap / 'config.json')
    (refs / 'main').write_text(snapshot)
    os.environ['HF_HUB_OFFLINE'] = '1'
    os.environ['TRANSFORMERS_OFFLINE'] = '1'
    return snap

def assert_resolver_locked(snapshot):
    from huggingface_hub import hf_hub_download
    expected = {
        'model.safetensors': ((snapshot / 'model.safetensors').resolve(), EXPECTED_WEIGHTS_SHA256),
        'config.json': ((snapshot / 'config.json').resolve(), EXPECTED_CONFIG_SHA256),
    }
    for filename, (expected_path, expected_digest) in expected.items():
        try:
            resolved = Path(hf_hub_download(repo_id=MODEL_ID, filename=filename)).resolve()
        except Exception as exc:
            raise RuntimeError('Verified snapshot is staged but Hugging Face cannot resolve it offline. Use Runtime → Restart session and run the notebook top-to-bottom.') from exc
        if resolved != expected_path:
            raise RuntimeError(f'Offline checkpoint lock is not in effect for {filename}.\nExpected: {expected_path}\nResolved: {resolved}\nUse Runtime → Restart session and run the notebook top-to-bottom.')
        resolved_digest = sha256_file(resolved)
        if resolved_digest != expected_digest:
            raise RuntimeError(f'Resolved {filename} digest changed after staging.\nExpected: {expected_digest}\nActual:   {resolved_digest}')
    print('✓ Hugging Face resolver locked to the verified offline snapshot and digests.')

MODEL_SOURCE = 'Pinned upstream'  # @param ['DIMER ZIP', 'Pinned upstream']

weights_path = MODEL_DIR / 'model.safetensors'
config_path = MODEL_DIR / 'config.json'

if MODEL_SOURCE == 'DIMER ZIP':
    weights_from_dimer(weights_path)
    fetch_pinned('config.json', config_path)
else:
    fetch_pinned('model.safetensors', weights_path)
    fetch_pinned('config.json', config_path)

weights_digest = verify(weights_path, EXPECTED_WEIGHTS_SHA256, 'model.safetensors')
verify(config_path, EXPECTED_CONFIG_SHA256, 'config.json')
SNAPSHOT_PATH = install_offline_snapshot(weights_path, config_path, weights_digest)
print('✓ Offline snapshot:', SNAPSHOT_PATH)
assert_resolver_locked(SNAPSHOT_PATH)


## 3. Choose a dataset

If you do not have a dataset, choose **Sample dataset (FreshRetailNet)**. The bundled ZIP contains `train.csv`, `val.csv`, and `test.csv`; the notebook preserves those leakage-aware partitions instead of randomly re-splitting them.

The sample has 4,180 training rows, 1,600 validation rows, 1,600 test rows, 17 features, and a continuous `sale_amount` target seven days ahead. It is derived from FreshRetailNet-50K and redistributed under **CC BY 4.0** for tutorial/smoke-test use, not benchmarking.

[Read the sample DATASET_CARD.md](https://github.com/kurtvalcorza/mitra-regressor-pipeline/blob/5625a9eeca94b8c72b9ad1ec78d07ecbaa720903/examples/sample-data/DATASET_CARD.md)

For your own data:

- **Upload CSV** creates a seeded random holdout and therefore assumes rows are approximately IID. Do **not** use this mode for time-dependent, grouped, panel, lagged, or rolling-window data unless random splitting is scientifically appropriate.
- **Upload pre-split train/val/test** preserves partitions you prepared externally, which is the safer choice for temporal/grouped data or any workflow with an embargo/purge rule. Feature columns may be in different orders; the notebook validates names and reorders validation/test columns to the training order.


In [ ]:
import csv
import io
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

NETWORK_TIMEOUT_SECONDS = 30

DATA_SOURCE = 'Sample dataset (FreshRetailNet)'  # @param ['Sample dataset (FreshRetailNet)', 'Upload CSV', 'Upload pre-split train/val/test']
TARGET_COLUMN = 'target'                         # @param {type:'string'}
DROP_COLUMNS = ''                                # @param {type:'string'}
VALIDATION_SPLIT = 0.20                          # @param {type:'number'}
SEED = 42                                        # @param {type:'integer'}

SAMPLE_REVISION = '5625a9eeca94b8c72b9ad1ec78d07ecbaa720903'
SAMPLE_ZIP_URL = f'https://raw.githubusercontent.com/kurtvalcorza/mitra-regressor-pipeline/{SAMPLE_REVISION}/examples/sample-data/freshretailnet-h7.zip'
SAMPLE_CARD_URL = f'https://github.com/kurtvalcorza/mitra-regressor-pipeline/blob/{SAMPLE_REVISION}/examples/sample-data/DATASET_CARD.md'

using_sample = DATA_SOURCE == 'Sample dataset (FreshRetailNet)'
using_presplit_data = DATA_SOURCE in {'Sample dataset (FreshRetailNet)', 'Upload pre-split train/val/test'}
test_data = None
TRAIN_ROW_CAP_APPLIED = False

def read_csv_payload(payload, label):
    text = payload.decode('utf-8-sig')
    rows = csv.reader(io.StringIO(text, newline=''))
    header = next((row for row in rows if row and not (len(row) == 1 and not row[0].strip())), [])
    seen = set()
    duplicates = []
    for name in header:
        if name in seen and name not in duplicates:
            duplicates.append(name)
        seen.add(name)
    if duplicates:
        raise ValueError(f'{label} contains duplicate column names: {duplicates}')
    return pd.read_csv(io.BytesIO(payload))

def read_presplit_upload():
    from google.colab import files
    uploaded = files.upload()
    by_base = {Path(name).name.lower(): payload for name, payload in uploaded.items()}
    required = {'train.csv', 'val.csv', 'test.csv'}
    missing = sorted(required - set(by_base))
    if missing:
        raise RuntimeError(f'Upload train.csv, val.csv, and test.csv together. Missing: {missing}')
    return (
        read_csv_payload(by_base['train.csv'], 'train.csv'),
        read_csv_payload(by_base['val.csv'], 'val.csv'),
        read_csv_payload(by_base['test.csv'], 'test.csv'),
    )

if using_sample:
    with urllib.request.urlopen(SAMPLE_ZIP_URL, timeout=NETWORK_TIMEOUT_SECONDS) as r:
        payload = r.read()
    with zipfile.ZipFile(io.BytesIO(payload)) as z:
        names = {Path(n).name: n for n in z.namelist() if not n.endswith('/')}
        required = {'train.csv', 'val.csv', 'test.csv'}
        missing = sorted(required - set(names))
        if missing:
            raise RuntimeError(f'Sample ZIP missing: {missing}')
        train_data = pd.read_csv(z.open(names['train.csv']))
        holdout_data = pd.read_csv(z.open(names['val.csv']))
        test_data = pd.read_csv(z.open(names['test.csv']))
    TARGET_COLUMN = 'target'
    print('✓ Using FreshRetailNet regression sample with preserved train/val/test splits.')
    print('  Sample revision:', SAMPLE_REVISION)
    print('  Dataset card:', SAMPLE_CARD_URL)
elif DATA_SOURCE == 'Upload pre-split train/val/test':
    train_data, holdout_data, test_data = read_presplit_upload()
    print('✓ Using uploaded train/val/test partitions without re-splitting.')
else:
    from google.colab import files
    uploaded = files.upload()
    csvs = [(name, payload) for name, payload in uploaded.items() if name.lower().endswith('.csv')]
    if len(csvs) != 1:
        raise RuntimeError('Upload exactly one labelled CSV.')
    data = read_csv_payload(csvs[0][1], 'uploaded CSV')
    print('⚠ Upload CSV uses a seeded random holdout and assumes rows are IID. For temporal/grouped/lagged data, use the pre-split option.')

drop_columns = [c.strip() for c in DROP_COLUMNS.split(',') if c.strip() and c.strip() != TARGET_COLUMN]

def prepare(df, name, require_variation=False, min_rows=2):
    if df.columns.duplicated().any():
        raise ValueError(f'{name}: duplicate column names are not supported.')
    if TARGET_COLUMN not in df.columns:
        raise ValueError(f'{name}: target {TARGET_COLUMN!r} not found.')
    out = df.drop(columns=[c for c in drop_columns if c in df.columns], errors='ignore').copy()
    raw_target = out[TARGET_COLUMN]
    numeric_target = pd.to_numeric(raw_target, errors='coerce')
    non_numeric = raw_target.notna() & numeric_target.isna()
    if non_numeric.any():
        examples = raw_target[non_numeric].astype(str).head(5).tolist()
        raise ValueError(f'{name}: target must be numeric; examples of invalid values: {examples}')
    out[TARGET_COLUMN] = numeric_target
    rows_before_target_drop = len(out)
    out = out.dropna(subset=[TARGET_COLUMN]).copy()
    dropped_target_rows = rows_before_target_drop - len(out)
    if dropped_target_rows:
        print(f'⚠ {name}: dropped {dropped_target_rows:,} row(s) with a missing target.')
    if not np.isfinite(out[TARGET_COLUMN].to_numpy(dtype=float)).all():
        raise ValueError(f'{name}: target contains infinite values; use finite numeric regression targets only.')
    duplicate_rows = int(out.duplicated().sum())
    if duplicate_rows:
        print(f'⚠ {name}: {duplicate_rows:,} exact duplicate labelled rows detected; inspect for leakage or accidental copies.')
    features = [c for c in out.columns if c != TARGET_COLUMN]
    errors = []
    if len(out) < min_rows:
        errors.append(f'use at least {min_rows} labelled rows')
    if not features:
        errors.append('no feature columns remain')
    if len(features) > 500:
        errors.append(f'{len(features)} features exceed the 500-feature limit')
    if require_variation and out[TARGET_COLUMN].nunique(dropna=True) < 2:
        errors.append('training target has no variation')
    if errors:
        raise ValueError(f'{name} is not ready: ' + '; '.join(errors))
    return out, features

if using_presplit_data:
    train_data, features = prepare(train_data, 'train.csv', require_variation=True, min_rows=50)
    holdout_data, val_features = prepare(holdout_data, 'val.csv')
    test_data, test_features = prepare(test_data, 'test.csv')
    train_feature_set = set(features)
    if set(val_features) != train_feature_set or set(test_features) != train_feature_set:
        raise ValueError('train/val/test feature column names do not match.')
    ordered_columns = features + [TARGET_COLUMN]
    holdout_data = holdout_data.reindex(columns=ordered_columns)
    test_data = test_data.reindex(columns=ordered_columns)
else:
    clean, features = prepare(data, 'uploaded CSV', require_variation=True, min_rows=50)
    if not 0.05 <= VALIDATION_SPLIT <= 0.40:
        raise ValueError('VALIDATION_SPLIT must be 0.05–0.40.')
    train_data, holdout_data = train_test_split(
        clean,
        test_size=VALIDATION_SPLIT,
        random_state=SEED,
        shuffle=True,
    )
    if train_data[TARGET_COLUMN].nunique(dropna=True) < 2:
        raise ValueError('Training split has no target variation. Use pre-split data, add more varied rows, or adjust VALIDATION_SPLIT.')

TRAIN_ROWS_BEFORE_CAP = len(train_data)
if len(train_data) > 10_000:
    train_data = train_data.sample(n=10_000, random_state=SEED)
    TRAIN_ROW_CAP_APPLIED = True
    if train_data[TARGET_COLUMN].nunique(dropna=True) < 2:
        raise ValueError('Capped training split has no target variation; provide a representative pre-split training set.')

FEATURE_COLUMNS = [c for c in train_data.columns if c != TARGET_COLUMN]
PROBLEM_TYPE = 'regression'

target_summary = pd.DataFrame({
    'split': ['train', 'holdout'] + (['test'] if test_data is not None else []),
    'rows': [len(train_data), len(holdout_data)] + ([len(test_data)] if test_data is not None else []),
    'target_mean': [train_data[TARGET_COLUMN].mean(), holdout_data[TARGET_COLUMN].mean()] + ([test_data[TARGET_COLUMN].mean()] if test_data is not None else []),
    'target_std': [train_data[TARGET_COLUMN].std(), holdout_data[TARGET_COLUMN].std()] + ([test_data[TARGET_COLUMN].std()] if test_data is not None else []),
    'target_min': [train_data[TARGET_COLUMN].min(), holdout_data[TARGET_COLUMN].min()] + ([test_data[TARGET_COLUMN].min()] if test_data is not None else []),
    'target_max': [train_data[TARGET_COLUMN].max(), holdout_data[TARGET_COLUMN].max()] + ([test_data[TARGET_COLUMN].max()] if test_data is not None else []),
})
display(pd.DataFrame({
    'Item': ['Training rows', 'Holdout rows', 'Independent test rows', 'Features', 'Target', 'Problem type'],
    'Value': [len(train_data), len(holdout_data), len(test_data) if test_data is not None else None, len(FEATURE_COLUMNS), TARGET_COLUMN, PROBLEM_TYPE],
}))
display(target_summary)

zero_fraction = float((train_data[TARGET_COLUMN] == 0).mean())
if zero_fraction >= 0.50:
    print(f'⚠ Training target is {zero_fraction:.1%} zeros. Compare Mitra against a strong naive baseline; highly intermittent targets can be difficult.')
if TRAIN_ROW_CAP_APPLIED:
    print(f'⚠ Training rows capped from {TRAIN_ROWS_BEFORE_CAP:,} to {len(train_data):,}.')
if len(train_data) > 5_000:
    print("⚠ Above Mitra's particularly strong reported ≤5,000-sample regime.")
if len(FEATURE_COLUMNS) > 100:
    print("⚠ Above Mitra's particularly strong reported ≤100-feature regime.")


## 4. Evaluate pretrained Mitra, then optionally fine-tune

`fine_tune=False` uses labelled examples as context without updating weights. Fine-tuning requires a GPU. When a pre-split source is used, `val.csv` is the holdout and `test.csv` is reported separately as an independent test set.

**Colab memory note:** `MAX_MEMORY_USAGE_RATIO=1.10` has now completed an end-to-end Mitra Regressor run on a standard Tesla T4. Memory is still tight: the observed fine-tune reduced `max_samples_support` from 8192 → 4096 → 2048 before completing under the time-limit path. Keep 1.10 as a cautious setting; do not increase it merely to suppress memory warnings. Values above 1.0 deliberately accept more OOM risk.

**Fine-tuning note:** `FINE_TUNE_STEPS=50` makes the requested schedule explicit. `FINE_TUNE_TIME_LIMIT` can truncate that schedule, so a time-limited run should not be interpreted as a controlled 50-step experiment.

**Metric direction:** AutoGluon stores regression error metrics in higher-is-better form, so `evaluate()` returns them negated. This notebook converts MAE/RMSE/MSE and related errors back to conventional positive values; R² and correlations remain higher-is-better.

**Rerun safety:** Step 4 clears predictor objects and output directories from any earlier execution before fitting, so a failed rerun cannot accidentally export an older model.


**Selection guard:** holdout/test splits may be as small as 2 rows for evaluation, but automatic pretrained-vs-fine-tuned artifact selection requires at least `MIN_SELECTION_HOLDOUT_ROWS=50` holdout rows. Below that threshold, the notebook keeps the pretrained predictor and reports fine-tuned metrics only as evidence.

In [ ]:
import gc
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from autogluon.tabular import TabularPredictor

EVAL_METRIC = 'mean_absolute_error'  # @param ['mean_absolute_error', 'root_mean_squared_error']
BASELINE_TIME_LIMIT = 300            # @param {type:'integer'}
RUN_FINE_TUNING = False              # @param {type:'boolean'}
FINE_TUNE_STEPS = 50                 # @param {type:'integer'}
FINE_TUNE_TIME_LIMIT = 600           # @param {type:'integer'}
MAX_MEMORY_USAGE_RATIO = 1.10        # @param {type:'number'}
MIN_SELECTION_HOLDOUT_ROWS = 50

BASELINE_PATH = Path('/content/mitra-baseline')
FINETUNED_PATH = Path('/content/mitra-finetuned')
EXPORT_ZIP_PATH = Path('/content/mitra-predictor.zip')

FIT_RUN_COMPLETED = False
for stale_name in (
    'active_predictor',
    'recommended_predictor',
    'active_mode',
    'selection_basis',
    'baseline_predictor',
    'finetuned_predictor',
    'baseline_metrics',
    'baseline_test_metrics',
    'finetuned_metrics',
    'finetuned_test_metrics',
):
    globals().pop(stale_name, None)

for stale_path in (BASELINE_PATH, FINETUNED_PATH):
    shutil.rmtree(stale_path, ignore_errors=True)
EXPORT_ZIP_PATH.unlink(missing_ok=True)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

baseline_predictor = None
finetuned_predictor = None
baseline_metrics = None
baseline_test_metrics = None
finetuned_metrics = None
finetuned_test_metrics = None

CUDA_AVAILABLE = torch.cuda.is_available()
print('PyTorch:', torch.__version__)
print('PyTorch CUDA build:', torch.version.cuda)
print('CUDA available:', CUDA_AVAILABLE, torch.cuda.get_device_name(0) if CUDA_AVAILABLE else '')
print(f'AutoGluon memory safety ratio: {MAX_MEMORY_USAGE_RATIO:.2f}')
print('✓ Cleared stale predictor state and output paths before fitting.')

REG_LOWER_IS_BETTER = {
    'root_mean_squared_error',
    'mean_squared_error',
    'mean_absolute_error',
    'median_absolute_error',
    'mean_absolute_percentage_error',
    'symmetric_mean_absolute_percentage_error',
    'root_mean_squared_logarithmic_error',
}
MITRA_METRIC_MAP = {
    'mean_absolute_error': 'mae',
    'root_mean_squared_error': 'rmse',
}
if EVAL_METRIC not in MITRA_METRIC_MAP:
    raise ValueError(
        f'Unsupported EVAL_METRIC {EVAL_METRIC!r}. Choose one of: {sorted(MITRA_METRIC_MAP)}'
    )

def seed_everything():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

def fit_mitra(fine_tune, path, time_limit, steps=None):
    seed_everything()
    hp = {'fine_tune': fine_tune, 'seed': SEED}
    native_metric = MITRA_METRIC_MAP.get(EVAL_METRIC)
    if native_metric is not None:
        hp['metric'] = native_metric
    if fine_tune:
        if steps is None or steps <= 0:
            raise ValueError('FINE_TUNE_STEPS must be a positive integer when fine-tuning is enabled.')
        hp['fine_tune_steps'] = steps
    predictor = TabularPredictor(
        label=TARGET_COLUMN,
        problem_type='regression',
        eval_metric=EVAL_METRIC,
        path=str(path),
        verbosity=2,
    )
    predictor.fit(
        train_data,
        hyperparameters={'MITRA': hp},
        fit_weighted_ensemble=False,
        time_limit=time_limit,
        ag_args_fit={'max_memory_usage_ratio': MAX_MEMORY_USAGE_RATIO},
    )
    if not any('mitra' in n.lower() for n in predictor.model_names()):
        raise RuntimeError(f'Expected Mitra; AutoGluon trained {predictor.model_names()}.')
    return predictor

def metrics(predictor, frame):
    raw = predictor.evaluate(frame, auxiliary_metrics=True, silent=True)
    return {k: float(-v if k in REG_LOWER_IS_BETTER else v) for k, v in raw.items()}

def metric_table(values, name):
    frame = pd.DataFrame({'value': pd.Series(values)})
    frame['direction'] = ['lower is better' if metric in REG_LOWER_IS_BETTER else 'higher is better' for metric in frame.index]
    frame.columns = [name, 'direction']
    return frame

baseline_predictor = fit_mitra(False, BASELINE_PATH, BASELINE_TIME_LIMIT)
baseline_metrics = metrics(baseline_predictor, holdout_data)
baseline_test_metrics = metrics(baseline_predictor, test_data) if test_data is not None else None
display(metric_table(baseline_metrics, 'Pretrained — holdout'))
if baseline_test_metrics is not None:
    display(metric_table(baseline_test_metrics, 'Pretrained — independent test'))

if RUN_FINE_TUNING:
    if not CUDA_AVAILABLE:
        raise RuntimeError('Fine-tuning requires a GPU. Choose Runtime → Change runtime type → GPU.')
    finetuned_predictor = fit_mitra(
        True,
        FINETUNED_PATH,
        FINE_TUNE_TIME_LIMIT,
        FINE_TUNE_STEPS,
    )
    finetuned_metrics = metrics(finetuned_predictor, holdout_data)
    comparison = pd.DataFrame({'Pretrained': baseline_metrics, 'Fine-tuned': finetuned_metrics})
    comparison['direction'] = ['lower is better' if metric in REG_LOWER_IS_BETTER else 'higher is better' for metric in comparison.index]
    display(comparison)
    print('Interpret small metric deltas cautiously; repeat across seeds/splits when the decision matters. The time limit may truncate the requested fine-tune schedule.')
    if test_data is not None:
        finetuned_test_metrics = metrics(finetuned_predictor, test_data)
        display(metric_table(finetuned_test_metrics, 'Fine-tuned — independent test'))
else:
    print('Fine-tuning skipped. Set RUN_FINE_TUNING=True on a GPU to run it.')

def metric_is_better(candidate, baseline, metric_name):
    if metric_name not in candidate or metric_name not in baseline:
        raise RuntimeError(f'Metric {metric_name!r} was not returned by AutoGluon; cannot select a predictor safely.')
    if metric_name in REG_LOWER_IS_BETTER:
        return candidate[metric_name] < baseline[metric_name]
    return candidate[metric_name] > baseline[metric_name]

def metric_is_worse(candidate, baseline, metric_name):
    if metric_name not in candidate or metric_name not in baseline:
        return False
    if metric_name in REG_LOWER_IS_BETTER:
        return candidate[metric_name] > baseline[metric_name]
    return candidate[metric_name] < baseline[metric_name]

recommended_predictor = baseline_predictor
active_mode = 'pretrained'
selection_basis = 'default:pretrained'
if finetuned_predictor is not None:
    if len(holdout_data) < MIN_SELECTION_HOLDOUT_ROWS:
        selection_basis = (
            f'default:pretrained; holdout-too-small:'
            f'{len(holdout_data)}<{MIN_SELECTION_HOLDOUT_ROWS}'
        )
        print(
            '⚠ Holdout is too small for automatic model selection '
            f'({len(holdout_data)} rows; minimum {MIN_SELECTION_HOLDOUT_ROWS}). '
            'Keeping the pretrained predictor for inference/export. '
            'Fine-tuned metrics are still reported as evaluation evidence.'
        )
    else:
        selection_basis = f'holdout:{EVAL_METRIC}'
        if metric_is_better(finetuned_metrics, baseline_metrics, EVAL_METRIC):
            recommended_predictor = finetuned_predictor
            active_mode = 'fine-tuned'
        print(
            f'✓ Recommended predictor for inference/export: {active_mode} '
            f'(selected by {EVAL_METRIC} on the holdout: '
            f'pretrained={baseline_metrics[EVAL_METRIC]:.6g}, fine-tuned={finetuned_metrics[EVAL_METRIC]:.6g}).'
        )
    if finetuned_test_metrics is not None and baseline_test_metrics is not None:
        degraded = [
            metric_name for metric_name in baseline_test_metrics
            if metric_name in finetuned_test_metrics
            and metric_is_worse(finetuned_test_metrics, baseline_test_metrics, metric_name)
        ]
        if degraded:
            details = ', '.join(
                f'{name}: {baseline_test_metrics[name]:.6g} → {finetuned_test_metrics[name]:.6g}'
                for name in degraded
            )
            print(
                '⚠ Fine-tuning produced mixed independent-test evidence. '
                f'These metrics worsened: {details}. Selection never uses the independent test; '
                'it remains evaluation evidence only.'
            )
else:
    print('✓ Recommended predictor for inference/export: pretrained (fine-tuning not run).')

active_predictor = recommended_predictor

FIT_RUN_COMPLETED = True
print('✓ Step 4 completed successfully; this run is eligible for inference/export.')


## 5. Predict new rows

Upload an unlabelled CSV with the same feature columns. For a pre-split dataset, `test.csv` was already scored above; this step is for genuinely new rows.


In [ ]:
import csv
import io

import pandas as pd

def read_inference_csv(payload):
    text = payload.decode('utf-8-sig')
    rows = csv.reader(io.StringIO(text, newline=''))
    header = next((row for row in rows if row and not (len(row) == 1 and not row[0].strip())), [])
    seen = set()
    duplicates = []
    for name in header:
        if name in seen and name not in duplicates:
            duplicates.append(name)
        seen.add(name)
    if duplicates:
        raise ValueError(f'Inference CSV contains duplicate column names: {duplicates}')
    return pd.read_csv(io.BytesIO(payload))

RUN_NEW_DATA_INFERENCE = False  # @param {type:'boolean'}

if RUN_NEW_DATA_INFERENCE:
    if not globals().get('FIT_RUN_COMPLETED', False) or baseline_predictor is None:
        raise RuntimeError('No predictor was successfully trained in this Step 4 execution. Run Step 4 successfully before inference.')
    from google.colab import files
    uploaded = files.upload()
    csvs = [(name, payload) for name, payload in uploaded.items() if name.lower().endswith('.csv')]
    if len(csvs) != 1:
        raise RuntimeError('Upload exactly one inference CSV.')
    new_data = read_inference_csv(csvs[0][1])
    if new_data.columns.duplicated().any():
        duplicates = list(new_data.columns[new_data.columns.duplicated()])
        raise ValueError(f'Inference CSV contains duplicate column names: {duplicates}')
    missing = [c for c in FEATURE_COLUMNS if c not in new_data.columns]
    if missing:
        raise ValueError(f'Inference CSV is missing required features: {missing}')
    if 'prediction' in new_data.columns:
        raise ValueError("Inference CSV already contains a 'prediction' column; rename or remove it before running inference.")
    X = new_data.reindex(columns=FEATURE_COLUMNS).copy()
    active = active_predictor
    pred = active.predict(X)
    out = new_data.copy()
    out['prediction'] = pred.to_numpy()
    out.to_csv('/content/predictions.csv', index=False)
    display(out.head())
    files.download('/content/predictions.csv')
else:
    print('Inference skipped.')


## 6. Export the reusable predictor

The AutoGluon `TabularPredictor` directory is the reusable trained artifact. The exported ZIP is not a replacement `model.safetensors`; extract it and load the directory with `TabularPredictor.load(path)`. For best compatibility, reload it with the same runtime recorded in `tutorial_run_metadata.json` (this tutorial pins `autogluon.tabular[mitra]==1.5.0`).

The export cell refuses to package a predictor unless the **current Step 4 execution** completed successfully. If fine-tuning ran, Step 4 recommends the predictor using the configured `EVAL_METRIC` on the holdout only; the independent test remains evaluation evidence and can trigger a mixed-performance warning without being used for model selection. The export cell packages that recommended predictor and prints the ZIP SHA-256 for later verification.


In [ ]:
import importlib.metadata as importlib_metadata
import json
import shutil
import sys
from datetime import datetime, timezone
from pathlib import Path

if not globals().get('FIT_RUN_COMPLETED', False) or baseline_predictor is None:
    raise RuntimeError(
        'No predictor was successfully trained in this Step 4 execution. '
        'Run Step 4 successfully before exporting.'
    )

active_path = Path(active_predictor.path)
if not active_path.exists() or not any('mitra' in name.lower() for name in active_predictor.model_names()):
    raise RuntimeError('The current predictor artifact is missing or does not contain Mitra; refusing to export.')

metadata = {
    'base_model': MODEL_ID,
    'base_model_revision': PINNED_REVISION,
    'weights_sha256': EXPECTED_WEIGHTS_SHA256,
    'config_sha256': EXPECTED_CONFIG_SHA256,
    'model_source': MODEL_SOURCE,
    'autogluon_version': importlib_metadata.version('autogluon.tabular'),
    'torch_version': torch.__version__,
    'torch_cuda_version': torch.version.cuda,
    'python_version': sys.version.split()[0],
    'cuda_available': torch.cuda.is_available(),
    'exported_at_utc': datetime.now(timezone.utc).isoformat(),
    'mode': active_mode,
    'selection_basis': selection_basis,
    'problem_type': 'regression',
    'target_column': TARGET_COLUMN,
    'features': FEATURE_COLUMNS,
    'seed': SEED,
    'data_source': DATA_SOURCE,
    'sample_revision': SAMPLE_REVISION if using_sample else None,
    'sample_dataset_card': SAMPLE_CARD_URL if using_sample else None,
    'train_rows_before_cap': TRAIN_ROWS_BEFORE_CAP,
    'train_rows_used': len(train_data),
    'train_row_cap_applied': TRAIN_ROW_CAP_APPLIED,
    'holdout_rows': len(holdout_data),
    'independent_test_rows': len(test_data) if test_data is not None else None,
    'eval_metric': EVAL_METRIC,
    'baseline_time_limit_seconds': BASELINE_TIME_LIMIT,
    'fine_tuning_requested': RUN_FINE_TUNING,
    'fine_tune_steps_requested': FINE_TUNE_STEPS if RUN_FINE_TUNING else None,
    'fine_tune_time_limit_seconds': FINE_TUNE_TIME_LIMIT if RUN_FINE_TUNING else None,
    'fine_tune_schedule_may_be_truncated_by_time_limit': bool(RUN_FINE_TUNING),
    'max_memory_usage_ratio': MAX_MEMORY_USAGE_RATIO,
    'holdout_metrics_pretrained': baseline_metrics,
    'holdout_metrics_finetuned': finetuned_metrics,
    'independent_test_metrics_pretrained': baseline_test_metrics,
    'independent_test_metrics_finetuned': finetuned_test_metrics,
    'ai_assistance': {
        'model_configuration': 'GPT-5.6 Sol High',
        'provider_client': 'OpenAI / ChatGPT',
        'agent_relay_role': 'Builder',
        'note': 'Attribution is provenance, not sign-off or independent verification.',
    },
}
(active_path / 'tutorial_run_metadata.json').write_text(json.dumps(metadata, indent=2))
Path('/content/mitra-predictor.zip').unlink(missing_ok=True)
archive = shutil.make_archive('/content/mitra-predictor', 'zip', root_dir=active_path)
archive_digest = sha256_file(archive)
print('✓ Predictor archive:', archive)
print('SHA-256 (save this if you will verify the ZIP later):', archive_digest)


## 7. Reload smoke test

Before treating the ZIP as reusable, this cell extracts the **exported archive** into a fresh directory, reloads it with `TabularPredictor.load(...)`, and confirms that numeric predictions match the in-memory predictor on a small holdout sample.

This checks the actual packaging boundary users will rely on later.


In [ ]:
import shutil
import stat
import zipfile
from pathlib import Path

RELOAD_DIR = Path('/content/mitra-predictor-reload')
if RELOAD_DIR.exists():
    shutil.rmtree(RELOAD_DIR)
RELOAD_DIR.mkdir(parents=True)

with zipfile.ZipFile(archive) as z:
    reload_root = RELOAD_DIR.resolve()
    for info in z.infolist():
        member = Path(info.filename)
        if member.is_absolute() or '..' in member.parts:
            raise RuntimeError(f'Unsafe archive member path during reload smoke test: {info.filename!r}')
        mode = (info.external_attr >> 16) & 0o170000
        if mode == stat.S_IFLNK:
            raise RuntimeError(f'Symlink entries are not allowed during reload smoke test: {info.filename!r}')
        target = (reload_root / member).resolve()
        if target != reload_root and reload_root not in target.parents:
            raise RuntimeError(f'Archive member escapes reload root: {info.filename!r}')
    z.extractall(RELOAD_DIR)

reloaded_predictor = TabularPredictor.load(str(RELOAD_DIR))
smoke_X = holdout_data[FEATURE_COLUMNS].head(5).copy()

expected_pred = active_predictor.predict(smoke_X).reset_index(drop=True)
reloaded_pred = reloaded_predictor.predict(smoke_X).reset_index(drop=True)
if not np.allclose(expected_pred.to_numpy(dtype=float), reloaded_pred.to_numpy(dtype=float), rtol=1e-6, atol=1e-8):
    raise RuntimeError('Reload smoke test failed: regression predictions changed after ZIP export/reload.')

print('✓ Exported predictor ZIP reloads successfully and reproduces smoke-test predictions.')


## AI use and provenance

This tutorial was developed with substantial AI assistance using **GPT-5.6 Sol High** under human direction and review.

- AI model/configuration: **GPT-5.6 Sol High**
- Provider/client: **OpenAI / ChatGPT**
- Agent Relay role: **Builder**
- Base-model developer: **AutoGluon team, Amazon Web Services (AWS)**
- DIMER role: distributor of the pinned `model.safetensors` artifact, not model developer

AI attribution is **provenance, not sign-off** and does not independently verify correctness.
